In [13]:
import numpy as np
import pandas as pd
import requests
from pathlib import Path 
import yaml
import re
import json

# Ngày kiểm tra mẫu
date='2025-12-13'

# Đọc config
config_path= f"../configs/London.yaml"
with Path(config_path).open("r", encoding="utf-8") as f:
    config = yaml.safe_load(f)
    
# Lọc listing
listing_loc = pd.read_csv(Path("../data/processed/listings-loc.csv"), dtype = {"listing_id": "str"}, usecols = ["listing_id", "analysis_area"])
calendar_path=Path(f'../data/raw/{date}/calendar.csv.gz')

In [14]:
# Chọn kích cỡ đọc và các cột cần xử lý 
chunk_size=500_000
use_cols=['listing_id', 'date', 'available', 'minimum_nights', 'maximum_nights']

In [ ]:
# Đọc theo cụm 
for chunk in pd.read_csv(calendar_path,compression='gzip',dtype={'listing_id':'str'},usecols=use_cols,chunksize=chunk_size):
    # Ghép với listing yêu cầu
    chunk_ghep=chunk.merge(listing_loc[["listing_id"]],on='listing_id',how='right')

    chunk_ghep["available"]=chunk_ghep["available"].map({'t':True,'f':False})

    # Xử lý thời gian 
    chunk_ghep['date']=pd.to_datetime(chunk_ghep['date'],format='%Y-%m-%d',errors="coerce")
    
    # Lọc dữ liệu thiếu,ngoại lai, trùng lặp:
    chunk_ghep['flag_date_missing']=chunk_ghep['date'].isna()
    chunk_ghep['flag_available'] = chunk_ghep['available'].isna()
    chunk_ghep['flag_min_max_outlier'] = (chunk_ghep['minimum_nights']>config['qa']['min_nights_cap']) | (chunk_ghep['minimum_nights']<0) | (chunk_ghep['maximum_nights']<0)
    chunk_ghep['flag_min_max_odd'] = (chunk_ghep['minimum_nights']>chunk_ghep['maximum_nights'])
    chunk_ghep['flag_duplicated']=chunk_ghep.duplicated(subset=['listing_id','date'], keep='first')